# Unit 8: TensorBoard — 深度学习可视化利器

## 学习目标
- 理解 TensorBoard 的核心功能和使用场景
- 掌握 `SummaryWriter` 的各种日志记录 API
- 学会可视化损失曲线、准确率、权重分布、梯度流
- 学会记录模型计算图和图像
- 掌握 `add_hparams` 进行超参数对比实验
- 在 Jupyter 中内嵌 TensorBoard
- 实战：用 TensorBoard 完整跟踪 CIFAR-10 训练

## 8.1 为什么需要 TensorBoard？

训练深度学习模型时你面临的问题：
- 损失曲线靠 `print` 看？几十个 epoch 后根本看不清趋势
- 模型结构复杂？靠想象画不出来
- 梯度消失了？不知道什么时候开始消失的
- 试了 10 组超参数？手动对比太痛苦

**TensorBoard 的解决方案**：一个统一的 Web 界面，实时可视化训练过程中的**一切**。

它原本是 TensorFlow 的一部分，但 PyTorch 通过 `torch.utils.tensorboard` 完美支持。

### 核心能力一览

| 功能 | API | 解决的问题 |
|------|-----|-----------|
| **Scalars** | `add_scalar` / `add_scalars` | 损失、准确率趋势 |
| **Histograms** | `add_histogram` | 权重/梯度分布变化 |
| **Graph** | `add_graph` | 可视化模型结构 |
| **Images** | `add_image` / `add_images` | 查看输入/输出/特征图 |
| **HParams** | `add_hparams` | 超参数对比实验 |
| **Embeddings** | `add_embedding` | 高维特征降维可视化 |
| **Text** | `add_text` | 记录文本摘要 |

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torch.utils.tensorboard import SummaryWriter
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
from tqdm import tqdm
from datetime import datetime

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

## 8.2 SummaryWriter 入门

`SummaryWriter` 是 TensorBoard 的日志记录器。它把数据写到 `log_dir` 目录，TensorBoard 从该目录读取并展示。

### 基础用法
```python
writer = SummaryWriter("runs/experiment_name")
writer.add_scalar("Loss/train", loss_value, global_step=epoch)
writer.close()  # 用完后关闭
```

### 目录命名规范
推荐使用时间戳或实验名区分不同 run：
```python
"runs/exp1_baseline"      # 手动命名
"runs/2024-01-01_14-30"   # 时间戳
"runs/{dataset}_{model}"  # 数据集+模型
```

In [ ]:
log_dir = Path("runs") / f"demo_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
writer = SummaryWriter(log_dir=str(log_dir))
print(f"Log directory: {log_dir}")

## 8.3 记录标量 (Scalars) — 损失和准确率

`add_scalar(tag, scalar_value, global_step)` 是最常用的 API，用于记录任何随时间变化的标量值。

### tag 命名技巧
使用 `/` 分隔可以创建层级分组：
- `"Loss/train"` — 训练损失，在 TensorBoard 中归入 "Loss" 组
- `"Loss/val"` — 验证损失
- `"Accuracy/train"` — 训练准确率
- `"LR"` — 学习率

### 同时记录多个标量
用 `add_scalars(main_tag, tag_scalar_dict, global_step)` 可以把多条曲线放到同一张图上对比。

In [ ]:
for step in range(100):
    train_loss = np.exp(-step / 30) + np.random.normal(0, 0.05)
    val_loss = np.exp(-step / 30) + 0.15 + np.random.normal(0, 0.03)
    train_acc = 0.1 + 0.85 * (1 - np.exp(-step / 30))
    val_acc = 0.1 + 0.75 * (1 - np.exp(-step / 30))

    writer.add_scalars("Loss", {"train": train_loss, "val": val_loss}, step)
    writer.add_scalars("Accuracy", {"train": train_acc, "val": val_acc}, step)

    lr = 0.001 * (1 - step / 100)
    writer.add_scalar("LR", lr, step)

print("Logged 100 steps of simulated scalars.  ✔")

## 8.4 记录直方图 (Histograms) — 权重与梯度

`add_histogram(tag, values, global_step)` 用于记录**张量的分布**。

### 关键用途
- **权重分布**：观察是否发生梯度消失/爆炸
- **梯度分布**：检查各层梯度流是否正常
- **激活值**：确认激活函数是否饱和（如 sigmoid 的梯度饱和）

### 小技巧
TensorBoard 的 Histogram 面板支持 **OFFSET** 模式，可以看到分布的**随时间变化**——这对于检测训练异常极其有用。

| 异常 | 权重分布特征 |
|------|-------------|
| **梯度消失** | 权重几乎不变，histogram 一致 |
| **梯度爆炸** | 权重值急剧增大，histogram 越来越宽 |
| **dead ReLU** | 大量权重集中在 0 附近 |

这段代码是深度学习训练中监控模型内部状态的标准范式，通常配合 TensorBoard 使用。它的核心作用是：在每个训练步（step）记录所有参数的权重分布和梯度分布直方图，以便可视化诊断训练过程。

In [ ]:
class DemoNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(100, 50)
        self.fc2 = nn.Linear(50, 10)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

model_demo = DemoNet()
opt_demo = optim.SGD(model_demo.parameters(), lr=0.01)
criterion = nn.CrossEntropyLoss()

for step in range(50):
    x = torch.randn(32, 100)
    y = torch.randint(0, 10, (32,))
    opt_demo.zero_grad()
    loss = criterion(model_demo(x), y)
    loss.backward()
    opt_demo.step()

    # per-batch
    for name, param in model_demo.named_parameters():
        writer.add_histogram(f"Weights/{name}", param.data, step)
        if param.grad is not None:
            writer.add_histogram(f"Gradients/{name}", param.grad, step)

print("Logged 50 steps of weight/gradient histograms.  ✔")

## 8.5 记录模型计算图 (Graph)

`add_graph(model, input_to_model)` 可以自动追踪并可视化 PyTorch 模型的**计算图结构**。

### 注意
- `input_to_model` 必须是**一个 tensor 或 tuple of tensors**
- graph 只在调用时记录一次，通常放在训练开始前
- 支持双击节点展开查看内部细节

PyTorch 使用的是动态计算图（Define-by-Run），图结构只在实际执行前向传播时才被构建。与 TensorFlow 1.x 的静态图不同，PyTorch 模型本身并不存储完整的图结构。因此：
- add_graph 内部会对 dummy_input 执行一次 trace（追踪） 前向传播
- 通过记录这次前向传播中所有算子的调用顺序和数据流向，重建出完整的计算图
- dummy_input 的形状必须与模型匹配（正如上一问验证的 1×3×32×32），否则 trace 会失败

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 16, 3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
        self.pool = nn.MaxPool2d(2)
        self.fc = nn.Linear(32 * 8 * 8, 10)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = self.pool(x)
        x = F.relu(self.conv2(x))
        x = self.pool(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x

demo_model = SimpleCNN()
dummy_input = torch.randn(2, 3, 32, 32)
demo_model_2 = SimpleCNN()
writer.add_graph(demo_model, dummy_input)
writer.add_graph(demo_model_2, dummy_input)
print("Model graph recorded to TensorBoard.  ✔")
print("Double-click nodes in the GRAPHS tab to expand.")

## 8.6 记录图像 (Images)

`add_image(tag, img_tensor, global_step)` 和 `add_images(tag, img_tensor, global_step)` 用于可视化图片。

### 典型使用场景
- 查看**原始输入** + **数据增强后**的样本
- 观察**特征图 (feature maps)**
- 查看**错误分类**的样本
- 显示**生成模型**的输出（如 GAN / VAE）

### 图像格式要求
TensorBoard 期望：
- `add_image`：`(C, H, W)` 或 `(batch, C, H, W)`
- `add_images`：`(N, C, H, W)`
- 像素值范围 [0, 1]（float）或 [0, 255]（uint8）

In [ ]:
cifar10_mean = (0.4914, 0.4822, 0.4465)
cifar10_std = (0.2470, 0.2435, 0.2616)

demo_ds = datasets.CIFAR10(root="data", train=True, download=True, transform=transforms.ToTensor())

images, labels = [], []
for i in range(8):
    img, lbl = demo_ds[i]
    images.append(img)
    labels.append(demo_ds.classes[lbl])

grid = torch.stack(images)
writer.add_images("CIFAR-10/Samples", grid, global_step=3)
print("Logged 8 CIFAR-10 sample images.  ✔")

## 8.7 记录超参数 (HParams)

`add_hparams(hparam_dict, metric_dict)` 是 TensorBoard 的实验管理利器。

### 功能
- 对比**不同超参数组合**下的最终结果
- 在 HParams 仪表板中**并行视图**查看
- 支持**平行坐标图**（Parallel Coordinates Plot）发现参数间的关系
- 支持**散点图矩阵**（Scatter Plot Matrix）

In [ ]:
hparams = {
    "lr": 0.001,
    "batch_size": 128,
    "optimizer": "Adam",
    "use_dropout": True,
    "num_filters": 32,
}

metrics = {
    "hparam/accuracy": 0.854,
    "hparam/loss": 0.423,
    "hparam/epochs_to_best": 18,
}

writer.add_hparams(hparams, metrics)
print("HParams recorded. Open HParams tab to see comparison dashboard.  ✔")

## 8.8 在 Jupyter 中内嵌 TensorBoard

无需打开命令行，直接在 Notebook 里启动 TensorBoard！

### 方式 1：%tensorboard magic（推荐）
```python
%load_ext tensorboard
%tensorboard --logdir runs
```

### 方式 2：Python API 启动
```python
from tensorboard import notebook
notebook.start("--logdir runs")
```

In [ ]:
print("┌" + "─" * 58 + "┐")
print("│  [1;33m在 Jupyter 中启动 TensorBoard[0m                                     │")
print("│                                                              │")
print("│  %load_ext tensorboard                                      │")
print("│  %tensorboard --logdir runs --port 6006 --bind_all          │")
print("│                                                              │")
print("│  [90m或者访问 http://localhost:6006[0m                                │")
print("└" + "─" * 58 + "┘")

## 8.9 TensorBoard 命令行使用

如果不在 Jupyter 中，也可以在终端启动：

```bash
tensorboard --logdir runs --port 6006 --bind_all
```

常用参数：

| 参数 | 说明 |
|------|------|
| `--logdir` | 日志目录路径 |
| `--port` | 端口号 (默认 6006) |
| `--bind_all` | 允许外部访问（服务器场景） |
| `--reload_interval` | 刷新间隔，默认 30 秒 |
| `--samples_per_plugin` | 每个插件的采样数限制 |

### 如何选择 `--logdir`

```bash
# 查看单个实验
tensorboard --logdir runs/exp1

# 同时查看多个实验（通过父目录）
tensorboard --logdir runs        # 所有 experiment 一起显示

# 跨机器访问
tensorboard --logdir runs --host 0.0.0.0 --port 6006
```